In [82]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Dense,
    Concatenate,
    Layer,
    MultiHeadAttention,
)
from tensorflow.keras.optimizers import Adam
from statsmodels.tsa.statespace.sarimax import SARIMAX
import xgboost as xgb
import warnings

warnings.filterwarnings("ignore")

In [83]:
df = pd.read_csv("final_.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# Handle missing values
df.fillna(method="ffill", inplace=True)
df.fillna(method="bfill", inplace=True)

# Select target and features
target = "AQI"
features = [
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "Ozone (µg/m³)",
    "NO2 (µg/m³)",
    "NO (µg/m³)",
    "SO2 (µg/m³)",
    "CO (mg/m³)",
    "NH3 (µg/m³)",
]
df = df[["Timestamp"] + features + [target]]

In [84]:
df.head(20)

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),Ozone (µg/m³),NO2 (µg/m³),NO (µg/m³),SO2 (µg/m³),CO (mg/m³),NH3 (µg/m³),AQI
0,2019-01-01,79.16,193.325,45.42,66.09,63.46,28.805,2.205,5.543333,194.0
1,2019-01-02,83.91,223.905,54.61,63.01,81.52,30.985,2.870,5.543333,180.0
2,2019-01-03,102.33,257.545,68.84,59.21,84.28,30.675,3.470,5.543333,267.0
3,2019-01-04,83.36,210.475,67.64,60.60,81.05,29.640,3.575,5.543333,223.0
4,2019-01-05,76.57,202.300,63.44,64.11,61.42,30.220,3.230,5.543333,178.0
5,2019-01-06,80.92,201.085,61.47,60.19,49.21,28.205,2.330,5.543333,192.0
6,2019-01-07,53.07,152.360,48.66,69.04,43.16,29.815,2.100,5.543333,146.0
7,2019-01-08,56.07,140.085,67.36,56.31,52.55,34.095,2.325,5.543333,161.0
8,2019-01-09,74.09,216.010,85.43,48.68,57.36,28.300,2.210,5.543333,154.0
9,2019-01-10,84.82,233.730,69.76,51.30,72.74,27.880,2.575,5.543333,219.0


In [85]:
# Normalize numerical columns
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

In [86]:
df.head(20)

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),Ozone (µg/m³),NO2 (µg/m³),NO (µg/m³),SO2 (µg/m³),CO (mg/m³),NH3 (µg/m³),AQI
0,2019-01-01,0.142485,0.193190,0.272722,0.221305,0.262006,0.155287,0.213870,0.013436,194.0
1,2019-01-02,0.151249,0.223775,0.328517,0.210583,0.338019,0.167349,0.278371,0.013436,180.0
2,2019-01-03,0.185232,0.257421,0.414911,0.197354,0.349636,0.165634,0.336566,0.013436,267.0
3,2019-01-04,0.150234,0.210343,0.407626,0.202193,0.336041,0.159907,0.346751,0.013436,223.0
4,2019-01-05,0.137707,0.202166,0.382126,0.214413,0.253420,0.163116,0.313288,0.013436,178.0
5,2019-01-06,0.145732,0.200951,0.370166,0.200766,0.202029,0.151967,0.225994,0.013436,192.0
6,2019-01-07,0.094350,0.152217,0.292393,0.231575,0.176565,0.160875,0.203686,0.013436,146.0
7,2019-01-08,0.099885,0.139940,0.405926,0.187258,0.216087,0.184557,0.225509,0.013436,161.0
8,2019-01-09,0.133131,0.215879,0.515634,0.160696,0.236331,0.152493,0.214355,0.013436,154.0
9,2019-01-10,0.152927,0.233602,0.420497,0.169817,0.301065,0.150169,0.249758,0.013436,219.0


In [87]:
# Normalize numerical columns
scaler_target = MinMaxScaler()
df[target] = scaler.fit_transform(df[[target]])

In [88]:
df.head(20)

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),Ozone (µg/m³),NO2 (µg/m³),NO (µg/m³),SO2 (µg/m³),CO (mg/m³),NH3 (µg/m³),AQI
0,2019-01-01,0.142485,0.193190,0.272722,0.221305,0.262006,0.155287,0.213870,0.013436,0.368421
1,2019-01-02,0.151249,0.223775,0.328517,0.210583,0.338019,0.167349,0.278371,0.013436,0.338947
2,2019-01-03,0.185232,0.257421,0.414911,0.197354,0.349636,0.165634,0.336566,0.013436,0.522105
3,2019-01-04,0.150234,0.210343,0.407626,0.202193,0.336041,0.159907,0.346751,0.013436,0.429474
4,2019-01-05,0.137707,0.202166,0.382126,0.214413,0.253420,0.163116,0.313288,0.013436,0.334737
5,2019-01-06,0.145732,0.200951,0.370166,0.200766,0.202029,0.151967,0.225994,0.013436,0.364211
6,2019-01-07,0.094350,0.152217,0.292393,0.231575,0.176565,0.160875,0.203686,0.013436,0.267368
7,2019-01-08,0.099885,0.139940,0.405926,0.187258,0.216087,0.184557,0.225509,0.013436,0.298947
8,2019-01-09,0.133131,0.215879,0.515634,0.160696,0.236331,0.152493,0.214355,0.013436,0.284211
9,2019-01-10,0.152927,0.233602,0.420497,0.169817,0.301065,0.150169,0.249758,0.013436,0.421053


In [89]:
# ============================================
# DETRENDING USING SARIMA (LINEAR COMPONENT)
# ============================================
def detrend_with_sarima(series, order=(1, 1, 1), seasonal_order=(1, 1, 1, 24)):
    model = SARIMAX(series, order=order, seasonal_order=seasonal_order)
    fit = model.fit(disp=False)
    fitted_values = fit.fittedvalues
    # Some fitted values may be NaN at the beginning; replace with zeros.
    fitted_values = np.nan_to_num(fitted_values)
    residuals = series - fitted_values
    residuals = np.nan_to_num(residuals)
    return fitted_values, residuals


# Extract linear components for each pollutant (features)
# linear_components = {}
# non_linear_components = {}

# for pollutant in features:
#     if df[pollutant].isnull().all():
#         continue
#     linear, non_linear = detrend_with_sarima(df[pollutant].values)
#     linear_components[pollutant] = linear
#    non_linear_components[pollutant] = non_linear
import pickle

with open("../linear_comp.pkl", "rb") as f:
    linear_components = pickle.load(f)
with open("../non_linear_comp.pkl", "rb") as f:
    non_linear_components = pickle.load(f)


# Replace the pollutant columns with their linear components
for pollutant in linear_components:
    df[pollutant] = linear_components[pollutant]

In [90]:
# ===============================
# PREPARE SEQUENCES FOR MODEL
# ===============================
# We create sliding windows for:
#   (a) Features (shape: [samples, seq_len, num_features])
#   (b) Time intervals (Δt; shape: [samples, seq_len, 1])
sequence_length = 10
X_features = []
X_time = []
y = []

# For each sliding window, compute time gaps (in seconds) between successive timestamps.
# For the first element in each window, we set Δt = 0.
timestamps = df["Timestamp"].reset_index(drop=True)
for i in range(len(df) - sequence_length):
    # Extract feature window
    feat_window = df[features].iloc[i : i + sequence_length].values
    X_features.append(feat_window)
    # Extract corresponding timestamps and compute delta times
    ts_window = pd.to_datetime(timestamps.iloc[i : i + sequence_length])
    # Compute differences (in seconds). For first element, use 0.
    dt = [0]
    dt += list(ts_window.diff().fillna(pd.Timedelta(seconds=0)).dt.days.values[1:])
    dt = np.array(dt).reshape(-1, 1).astype("float32")
    X_time.append(dt)
    # Target is the AQI at time (i + sequence_length)
    y.append(df[target].iloc[i + sequence_length])

X_features = np.array(X_features)  # Shape: (samples, seq_len, num_features)
X_time = np.array(X_time)  # Shape: (samples, seq_len, 1)
y = np.array(y)

In [91]:
# Split into training and validation sets (without shuffling to preserve time order)
(X_feat_train, X_feat_val, X_time_train, X_time_val, y_train, y_val) = train_test_split(
    X_features, X_time, y, test_size=0.2, random_state=42, shuffle=False
)

In [92]:
class TLSTMCell(tf.keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super(TLSTMCell, self).__init__(**kwargs)
        self.units = units
        self.lstm_cell = tf.keras.layers.LSTMCell(units)

    @property
    def state_size(self):
        return self.lstm_cell.state_size

    @property
    def output_size(self):
        return self.lstm_cell.output_size

    def call(self, inputs, states, training=None):
        # Assume inputs shape: (batch, features + 1) where the last channel is delta_t.
        feature_dim = tf.shape(inputs)[-1] - 1
        features = inputs[:, :feature_dim]
        delta_t = inputs[:, feature_dim:]
        # Apply time decay to features
        adjusted_features = features * tf.exp(-delta_t)
        return self.lstm_cell(adjusted_features, states, training=training)

    def get_config(self):
        config = super(TLSTMCell, self).get_config()
        config.update(
            {
                "units": self.units,
            }
        )
        return config

In [93]:
# ===============================
# MODEL DEFINITION
# ===============================
def create_tlstm_model(lr, tlstm_units, kernel_size, feature_shape, time_shape):
    """
    feature_shape: (seq_len, num_features)
    time_shape: (seq_len, 1)
    """
    # Two inputs: one for features and one for time intervals.
    input_features = Input(shape=feature_shape, name="input_features")
    input_time = Input(shape=time_shape, name="input_time")

    # Process feature input through a Conv1D layer and pooling.
    x = Conv1D(filters=64, kernel_size=kernel_size, activation="relu", padding="same")(
        input_features
    )
    x = MaxPooling1D(pool_size=2)(x)
    #x = AttentionLayer(num_heads=4, key_dim=64)(x)

    # Downsample the time input similarly so that both streams align.
    t = MaxPooling1D(pool_size=2)(input_time)

    # Concatenate along the feature axis.
    combined = Concatenate(axis=-1)([x, t])
    # The resulting sequence length will be reduced relative to the original.

    # Use a Bidirectional TLSTM.
    # Wrap our custom TLSTM cell with the RNN layer.
    tlstm_cell = TLSTMCell(tlstm_units)
    rnn_layer = tf.keras.layers.RNN(tlstm_cell, return_sequences=False)
    bi_rnn = tf.keras.layers.Bidirectional(rnn_layer)(combined)

    outputs = Dense(1)(bi_rnn)
    model = Model(inputs=[input_features, input_time], outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=lr), loss="mean_squared_error")
    return model

In [94]:
lr, tlstm_units, kernel_size = [0.0019009703397010183, 28, 4]
# ===============================
# TRAIN FINAL MODEL
# ===============================
final_model = create_tlstm_model(
    lr,
    int(tlstm_units),
    int(kernel_size),
    feature_shape=X_feat_train.shape[1:],
    time_shape=X_time_train.shape[1:],
)
final_model.fit(
    [X_feat_train, X_time_train],
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=([X_feat_val, X_time_val], y_val),
)
pred_tlstm = final_model.predict([X_feat_val, X_time_val])

Epoch 1/30
457/457 [==============================] - 4s 5ms/step - loss: 0.0077 - val_loss: 0.0113
Epoch 2/30
457/457 [==============================] - 2s 4ms/step - loss: 0.0059 - val_loss: 0.0092
Epoch 3/30
457/457 [==============================] - 2s 4ms/step - loss: 0.0057 - val_loss: 0.0090
Epoch 4/30
457/457 [==============================] - 2s 4ms/step - loss: 0.0057 - val_loss: 0.0099
Epoch 5/30
457/457 [==============================] - 2s 4ms/step - loss: 0.0056 - val_loss: 0.0089
Epoch 6/30
457/457 [==============================] - 2s 4ms/step - loss: 0.0056 - val_loss: 0.0103
Epoch 7/30
457/457 [==============================] - 2s 4ms/step - loss: 0.0056 - val_loss: 0.0086
Epoch 8/30
457/457 [==============================] - 2s 5ms/step - loss: 0.0055 - val_loss: 0.0092
Epoch 9/30
457/457 [==============================] - 3s 7ms/step - loss: 0.0055 - val_loss: 0.0086
Epoch 10/30
457/457 [==============================] - 3s 6ms/step - loss: 0.0055 - val_loss: 0.0109

In [95]:
# ===============================
# XGBOOST BOOSTING STAGE
# ===============================
# For combining, we extract the linear components for a few pollutants.
# (For example, using PM2.5, PM10, NO2, and CO if available.)
# Ensure these components align with the prediction period.
num_val = len(pred_tlstm)
# Adjust names if needed; here we assume the keys are as in features.
linear_pm25 = linear_components["PM2.5 (µg/m³)"][-num_val:].reshape(-1, 1)
linear_pm10 = linear_components["PM10 (µg/m³)"][-num_val:].reshape(-1, 1)
linear_ozone = linear_components["Ozone (µg/m³)"][-num_val:].reshape(-1, 1)
linear_no2 = linear_components["NO2 (µg/m³)"][-num_val:].reshape(-1, 1)
linear_no = linear_components["NO (µg/m³)"][-num_val:].reshape(-1, 1)
linear_so2 = linear_components["SO2 (µg/m³)"][-num_val:].reshape(-1, 1)
linear_co = linear_components["CO (mg/m³)"][-num_val:].reshape(-1, 1)
linear_nh3 = linear_components["NH3 (µg/m³)"][-num_val:].reshape(-1, 1)

# Combine all 8 pollutant linear components with the TLSTM prediction
combined_features = np.concatenate(
    [linear_pm25, linear_pm10, linear_ozone, linear_no2,
     linear_no, linear_so2, linear_co, linear_nh3, pred_tlstm],
    axis=1
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
}

In [96]:
xgb_model = xgb.XGBRegressor()
xgb_model.fit(combined_features, y_val)
final_predictions = xgb_model.predict(combined_features)
print(f"MSE: {mean_squared_error(y_val, final_predictions)}")
print(f"MAE: {mean_absolute_error(y_val, final_predictions)}")
print(f"R2 Score: {r2_score(y_val, final_predictions)}")
rmse = np.sqrt(mean_squared_error(y_val, final_predictions))
print(f"RMSE: {rmse}")
# MSE: 0.00024648878867744134
# MAE: 0.011292530501851975
# R2 Score: 0.9946993459740087
# RMSE: 0.015699961422801056

MSE: 0.00024125803606203
MAE: 0.011052905086379582
R2 Score: 0.9948118314548234
RMSE: 0.015532483254844668


In [97]:
y_train.shape , X_feat_train.shape

((14600,), (14600, 10, 8))

In [98]:
y_train[0:3]

array([0.40631579, 0.38105263, 0.39368421])

In [99]:
df.head(20)

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),Ozone (µg/m³),NO2 (µg/m³),NO (µg/m³),SO2 (µg/m³),CO (mg/m³),NH3 (µg/m³),AQI
0,2019-01-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.368421
1,2019-01-02,0.142485,0.193190,0.272722,0.221305,0.262006,0.155287,0.213870,0.013436,0.338947
2,2019-01-03,0.151249,0.223775,0.328517,0.210583,0.338019,0.167349,0.278371,0.013436,0.522105
3,2019-01-04,0.185232,0.257421,0.414911,0.197354,0.349636,0.165634,0.336566,0.013436,0.429474
4,2019-01-05,0.150234,0.210343,0.407626,0.202193,0.336041,0.159907,0.346751,0.013436,0.334737
5,2019-01-06,0.137707,0.202166,0.382126,0.214413,0.253420,0.163116,0.313288,0.013436,0.364211
6,2019-01-07,0.145732,0.200951,0.370166,0.200766,0.202029,0.151967,0.225994,0.013436,0.267368
7,2019-01-08,0.094350,0.152217,0.292393,0.231575,0.176565,0.160875,0.203686,0.013436,0.298947
8,2019-01-09,0.099885,0.139940,0.405926,0.187258,0.216087,0.184557,0.225509,0.013436,0.284211
9,2019-01-10,0.133131,0.215879,0.515634,0.160696,0.236331,0.152493,0.214355,0.013436,0.421053


In [128]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import load_model

df_test = pd.read_csv("final_.csv")
df_test["Timestamp"] = pd.to_datetime(df_test["Timestamp"])
# Handle missing values
df_test.fillna(method="ffill", inplace=True)
df_test.fillna(method="bfill", inplace=True)

# Select target and features
target = "AQI"
features = [
    "PM2.5 (µg/m³)",
    "PM10 (µg/m³)",
    "Ozone (µg/m³)",
    "NO2 (µg/m³)",
    "NO (µg/m³)",
    "SO2 (µg/m³)",
    "CO (mg/m³)",
    "NH3 (µg/m³)",
]
df_test = df_test[["Timestamp"] + features + [target]]
df_test=df_test[4:14]
df_test

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),Ozone (µg/m³),NO2 (µg/m³),NO (µg/m³),SO2 (µg/m³),CO (mg/m³),NH3 (µg/m³),AQI
4,2019-01-05,76.57,202.300,63.44,64.11,61.42,30.220,3.230,5.543333,178.0
5,2019-01-06,80.92,201.085,61.47,60.19,49.21,28.205,2.330,5.543333,192.0
6,2019-01-07,53.07,152.360,48.66,69.04,43.16,29.815,2.100,5.543333,146.0
7,2019-01-08,56.07,140.085,67.36,56.31,52.55,34.095,2.325,5.543333,161.0
8,2019-01-09,74.09,216.010,85.43,48.68,57.36,28.300,2.210,5.543333,154.0
9,2019-01-10,84.82,233.730,69.76,51.30,72.74,27.880,2.575,5.543333,219.0
10,2019-01-11,83.62,243.860,71.35,46.42,75.99,34.040,2.615,5.543333,212.0
11,2019-01-12,78.65,238.240,79.40,49.48,57.38,34.840,2.490,5.543333,200.0
12,2019-01-13,74.26,206.340,75.82,34.15,62.04,34.495,2.615,5.543333,206.0
13,2019-01-14,66.22,177.290,72.51,44.56,36.98,32.490,2.320,5.543333,163.0


In [129]:
with open("./training_data/scaler_features.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
    df_test[features] = loaded_scaler.fit_transform(df_test[features])
    
with open("./training_data/scaler_target.pkl", "rb") as f:
    loaded_scaler = pickle.load(f)
    df_test[target] = loaded_scaler.fit_transform(df_test[[target]])
# Prepare the last 10-day input window
X_features = np.array(df_test[features].iloc[-10:].values).reshape(1, 10, len(features))

# Compute time intervals (Δt in days)
timestamps = df_test["Timestamp"].reset_index(drop=True)
ts_window = pd.to_datetime(timestamps.iloc[-10:])
dt = [0]  # First time difference is always 0
dt += list(ts_window.diff().fillna(pd.Timedelta(days=0)).dt.days.values[1:])
X_time = np.array(dt).reshape(1, 10, 1).astype("float32")


model = load_model("training_data/model", custom_objects={"TLSTMCell": TLSTMCell})

# Predict AQI for the next day
predicted_scaled_aqi = model.predict([X_features, X_time])

# Convert back to original AQI scale
# predicted_aqi = scaler1.inverse_transform([[0] * len(features) + [predicted_scaled_aqi[0][0]]])[0][-1]
print(f"Predicted AQI for Next Day: {predicted_scaled_aqi}")


1/1 [==============================] - 0s 251ms/step
Predicted AQI for Next Day: [[0.33392602]]


In [130]:
scaler_target.inverse_transform(predicted_scaled_aqi)

array([[170.3766]], dtype=float32)